# RAG Query: Before and After

This notebook demonstrates the value of RAG by querying an LLM:

1. **Without RAG** -- the LLM answers from its training data alone (often wrong or vague)
2. **With RAG** -- the LLM receives relevant document chunks from Milvus as context (accurate, sourced)

## Prerequisites

- Milvus running with a populated `rag_documents` collection (run `rag_ingestion.ipynb` first)
- A vLLM inference endpoint serving a model (e.g. Mistral 7B Instruct)
- `pymilvus`, `sentence-transformers`, `openai`, and `logfire` installed locally
- Optional: set `LOGFIRE_TOKEN` in your environment for Logfire traces (query notebook)


---
## Step 1 -- Install Dependencies


In [1]:
%pip install -q pymilvus sentence-transformers openai logfire

Note: you may need to restart the kernel to use updated packages.


---
## Step 2 -- Configure

Set the Milvus and LLM connection details. For local access via port-forward:

```bash
oc port-forward svc/milvus-milvus 19530:19530 -n milvus
```


In [4]:
# ---------------------------------------------------------------------------
# Milvus
# ---------------------------------------------------------------------------
MILVUS_HOST = "localhost"  # via: oc port-forward svc/milvus-milvus 19530:19530 -n milvus
MILVUS_PORT = 19530
MILVUS_DB = "default"
COLLECTION_NAME = "rag_documents"

# ---------------------------------------------------------------------------
# Embedding (must match the model used during ingestion)
# ---------------------------------------------------------------------------
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ---------------------------------------------------------------------------
# LLM inference endpoint (vLLM served via KServe)
# ---------------------------------------------------------------------------
INFERENCE_URL = "http://localhost:8080/v1"  # via: oc port-forward svc/mistral-7b-instruct-predictor 8080:80 -n rag-example
MODEL_NAME = "mistral-7b-instruct"

# ---------------------------------------------------------------------------
# RAG settings
# ---------------------------------------------------------------------------
TOP_K = 5  # Number of chunks to retrieve

print(f"Milvus:     {MILVUS_HOST}:{MILVUS_PORT}/{MILVUS_DB}.{COLLECTION_NAME}")
print(f"Embedding:  {EMBEDDING_MODEL}")
print(f"LLM:        {INFERENCE_URL} ({MODEL_NAME})")
print(f"Top-K:      {TOP_K}")

Milvus:     localhost:19530/default.rag_documents
Embedding:  sentence-transformers/all-MiniLM-L6-v2
LLM:        http://localhost:8080/v1 (mistral-7b-instruct)
Top-K:      5


---
## Step 3 -- Initialize Clients

Connect to Milvus, load the embedding model, and set up the LLM client.


In [5]:
import logfire
from openai import OpenAI
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

logfire.configure(service_name="rag-query")

# Milvus client
milvus = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)

# Verify collection exists
if milvus.has_collection(COLLECTION_NAME):
    milvus.load_collection(COLLECTION_NAME)
    stats = milvus.get_collection_stats(COLLECTION_NAME)
    print(f"Collection '{COLLECTION_NAME}' has {stats.get('row_count', '?')} vectors")
else:
    raise RuntimeError(f"Collection '{COLLECTION_NAME}' not found. Run rag_ingestion.ipynb first.")

# Embedding model (same one used during ingestion)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding model loaded: {EMBEDDING_MODEL}")

# LLM client (OpenAI-compatible API)
llm = OpenAI(base_url=INFERENCE_URL, api_key="unused")
print(f"LLM client ready: {MODEL_NAME}")

/opt/homebrew/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/homebrew/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Collection 'rag_documents' has 61840 vectors


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
LLM client ready: mistral-7b-instruct


---
## Step 4 -- Define Helper Functions


In [6]:
def ask_llm(question: str, context: str = "") -> str:
    """Send a question to the LLM, optionally with RAG context."""
    mode = "with-rag" if context else "without-rag"
    with logfire.span("llm-query {mode}", mode=mode, model=MODEL_NAME) as span:
        if context:
            prompt = (
                "You are a helpful assistant. Answer the user's question based on the "
                "provided context documents. If the answer is not in the context, say so.\n\n"
                f"## Context\n\n{context}\n\n"
                f"## Question\n\n{question}\n\n"
                "## Answer\n\n"
            )
        else:
            prompt = question

        response = llm.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.1,
        )
        answer = response.choices[0].message.content
        span.set_attribute("prompt_tokens", response.usage.prompt_tokens)
        span.set_attribute("completion_tokens", response.usage.completion_tokens)
        span.set_attribute("answer_length", len(answer))
    return answer


def search_milvus(question: str, top_k: int = TOP_K) -> list:
    """Embed the question and search Milvus for similar chunks."""
    with logfire.span("milvus-search", question=question, top_k=top_k,
                      collection=COLLECTION_NAME) as span:
        query_embedding = embed_model.encode([question], normalize_embeddings=True).tolist()

        results = milvus.search(
            collection_name=COLLECTION_NAME,
            data=query_embedding,
            limit=top_k,
            output_fields=["source_file", "chunk_index", "text"],
            search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
        )

        contexts = []
        for hits in results:
            for hit in hits:
                contexts.append({
                    "text": hit["entity"]["text"],
                    "source_file": hit["entity"]["source_file"],
                    "chunk_index": hit["entity"]["chunk_index"],
                    "score": hit["distance"],
                })

        span.set_attribute("results_count", len(contexts))
        if contexts:
            span.set_attribute("top_score", round(contexts[0]["score"], 3))
            span.set_attribute("top_source", contexts[0]["source_file"])
    return contexts


def build_context(chunks: list) -> str:
    """Format retrieved chunks into a context string for the prompt."""
    return "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"score: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )


print("Helper functions ready.")

Helper functions ready.


---
## Step 5 -- Pick a Question

Choose a question that the LLM is unlikely to answer correctly from its
training data alone -- something specific to the documents you ingested.


In [18]:
QUESTION = "Does fine-tuning smaller LLMs with additional career data surpass the performance of fine-tuning larger models?"

print(f"Question: {QUESTION}")

Question: When did the Bering Land Bridge become a national preserve?


---
## Step 6 -- Query WITHOUT RAG (Baseline)

Ask the LLM directly, with no retrieved context. The model must rely
entirely on its training data.


In [19]:
print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("ANSWER (without RAG)")
print("=" * 60)

answer_no_rag = ask_llm(QUESTION)
print(answer_no_rag)

Question: When did the Bering Land Bridge become a national preserve?

ANSWER (without RAG)
 The Bering Land Bridge, more accurately referred to as the Beringia, is not a national preserve. Instead, it's a region that connected Asia and North America during the last Ice Age. The concept of a Beringia National Preserve has been proposed but not yet established. However, there are several protected areas in the region, such as the Wrangell-St. Elias National Park and Preserve, Gates of the Arctic National Park and Preserve, and the Yukon-Charley Rivers National Preserve, which together are known as the Beringia National Parks. These parks were established between 1978 and 1980.


---
## Step 7 -- Retrieve Context from Milvus

Embed the question using the same model that was used during ingestion,
then search Milvus for the most similar document chunks.


In [14]:
chunks = search_milvus(QUESTION)

print(f"Retrieved {len(chunks)} chunks from Milvus:\n")
for i, c in enumerate(chunks, 1):
    print(f"  {i}. {c['source_file']} (chunk {c['chunk_index']}, score: {c['score']:.3f})")
    print(f"     {c['text'][:120]}...\n")

Retrieved 5 chunks from Milvus:

  1. 10000048021694510645.pdf (chunk 11, score: 0.650)
     Section 1.  MEMBERS - Membership in the Working Group shall be available to any person who is interested in Tennessee ba...

  2. 10000048374779037692.pdf (chunk 4, score: 0.642)
     There were enough members present for a quorum.
- V. APPROVAL OF MINUTES     -   Public Hearing - February 18, 2020
Regu...

  3. 10000048021694510645.pdf (chunk 19, score: 0.636)
     location agreed upon by the Working Group membership.  Other meetings of the officers or membership at large may be call...

  4. 10000048021694510645.pdf (chunk 6, score: 0.616)
     Section 1.  NAME - The name of this organization shall be the 'Tennessee Bat Working Group,' hereafter referred to as th...

  5. 10000048021694510645.pdf (chunk 12, score: 0.577)
     Section 1.  OFFICERS - Working Group officers shall consist of a Chair, an immediate Past Chair (except for the first te...



---
## Step 8 -- Query WITH RAG

Ask the same question, but this time include the retrieved document
chunks as context in the prompt.


In [15]:
context = build_context(chunks)

print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("ANSWER (with RAG)")
print("=" * 60)

answer_with_rag = ask_llm(QUESTION, context=context)
print(answer_with_rag)

Question: What is the quorum requirement for Tennessee Bat Working Group meetings?

ANSWER (with RAG)
 The quorum requirement for meetings of the Tennessee Bat Working Group is 15 percent of the membership or five members in good standing, whichever is greater. (Source: 10000048021694510645.pdf, chunk 19)


---
## Step 9 -- Compare Side by Side


In [16]:
print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("WITHOUT RAG")
print("=" * 60)
print(answer_no_rag)

print("\n" + "=" * 60)
print("WITH RAG")
print("=" * 60)
print(answer_with_rag)

print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)
for c in chunks:
    print(f"  - {c['source_file']} (chunk {c['chunk_index']}, score: {c['score']:.3f})")

Question: What is the quorum requirement for Tennessee Bat Working Group meetings?

WITHOUT RAG
 I don't have real-time data or specific details about the Tennessee Bat Working Group. However, I can tell you that the quorum requirement for a meeting is typically defined by the group's bylaws or rules of procedure. It's a number of members that must be present for the meeting to be valid and for decisions to be made. In the absence of specific information, you may want to refer to the group's official website or contact them directly for the accurate quorum requirement.

WITH RAG
 The quorum requirement for meetings of the Tennessee Bat Working Group is 15 percent of the membership or five members in good standing, whichever is greater. (Source: 10000048021694510645.pdf, chunk 19)

SOURCES
  - 10000048021694510645.pdf (chunk 11, score: 0.650)
  - 10000048374779037692.pdf (chunk 4, score: 0.642)
  - 10000048021694510645.pdf (chunk 19, score: 0.636)
  - 10000048021694510645.pdf (chunk 6, 

---
## Try Another Question

Change the question below and re-run to try a different query.


In [17]:
NEW_QUESTION = "Is the accuracy of NVILA compromised for efficiency gains?"

# Without RAG
print(f"Question: {NEW_QUESTION}")
print("\n--- Without RAG ---")
print(ask_llm(NEW_QUESTION))

# With RAG
new_chunks = search_milvus(NEW_QUESTION)
new_context = build_context(new_chunks)
print("\n--- With RAG ---")
print(ask_llm(NEW_QUESTION, context=new_context))

print("\n--- Sources ---")
for c in new_chunks:
    print(f"  - {c['source_file']} (chunk {c['chunk_index']}, score: {c['score']:.3f})")

Question: Which federal agencies are involved in bat conservation in the United States?

--- Without RAG ---
 Several federal agencies in the United States are involved in bat conservation due to their importance in ecosystems and as pollinators and pest controllers. Here are some key agencies:

1. U.S. Fish and Wildlife Service (USFWS): The USFWS is the primary federal agency responsible for the conservation, protection, and recovery of endangered and threatened species, including several bat species. They also manage the National Wildlife Refuge System, which provides critical habitat for bats.

2. National Park Service (NPS): The NPS manages national parks and monuments, many of which provide important bat habitats. They work to conserve bat populations and their habitats within these protected areas.

3. U.S. Forest Service (USFS): The USFS manages national forests and grasslands, which often provide essential roosting and foraging habitats for bats. They work to protect bat popula